In [ ]:
! git clone "https://github.com/fraco03/DR_RETFound.git"
%cd "/kaggle/working/DR_RETFound"
! pwd

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import wandb
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import ConcatDataset, DataLoader, WeightedRandomSampler
from src.utils import visualizza_attenzione_retfound
from src.dataset import RetinopathyDataset, get_transforms
from src.loss import SmoothL1Loss
from src.model_setup import build_retfound_regression

In [ ]:
def build_config():
    return {
        'idrid_images_dir': '/kaggle/input/idriv2/IDRiD_2.0/IDRiD_2.0/images',
        'idrid_csv_path': '/kaggle/input/idriv2/IDRiD_2.0/IDRiD_2.0.csv',
        'weights_path': '/kaggle/input/models/fraco0344/retfound/pytorch/default/1/RETFound_mae_natureCFP.pth',
        'checkpoint_dir': 'weights',
        'test_size': 0.2,
        'random_state': 42,
        'batch_size': 32,
        'num_workers': 4,
        'epochs': 20,
        'lr': 1e-4,
        'weight_decay': 1e-4,
        'loss_beta': 1.0,
        'sampler': 'weighted',
        'num_classes': 5,
        'log_confusion_matrix': True,
        'use_wandb': True,
        'wandb_project': 'dr-retfound',
        'wandb_run_name': None,
        'patience': 2,
        'early_stop_patience': 5,
        'clahe': True,
    }

cfg = build_config()
cfg

In [ ]:
idrid_df = pd.read_csv(cfg['idrid_csv_path'])
idrid_df['image_path'] = idrid_df['image_id'].apply(lambda x: os.path.join(cfg['idrid_images_dir'], f"{x}.jpg"))

idrid_transforms = get_transforms(clahe=cfg['clahe'], is_training=False)
idrid_dataset = RetinopathyDataset(
    idrid_df, 
    cfg['num_classes'], 
    transforms=idrid_transforms)

In [ ]:
test_loader = DataLoader(idrid_dataset, batch_size=cfg['batch_size'], shuffle=False, num_workers=cfg['num_workers'])

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_retfound_regression(cfg['weights_path'], device=device)
model = model.to(device)
model.eval()

In [ ]:
model.eval()
model_device = next(model.parameters()).device
all_preds = []
all_labels = []

print("--- Avvio Inferenza su Validation Set ---")
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(model_device)
        
        # Otteniamo l'output continuo (regressione)
        outputs = model(images)
        
        # Se il modello restituisce un tensore con più dimensioni, lo appiattiamo
        preds = outputs.squeeze().cpu().numpy()
        targets = labels.cpu().numpy()
        
        # Gestione caso batch_size=1
        if preds.ndim == 0:
            all_preds.append(float(preds))
            all_labels.append(float(targets))
        else:
            all_preds.extend(preds)
            all_labels.extend(targets)

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score
from src.utils import apply_thresholds

kappa_optimized_threshold = [0.5, 1.5, 2.5, 3.5] #modify
false_positive_optimized_threshold = [0.5, 1.5, 2.5, 3.5] #modify

classic_preds = all_preds.round().clamp(0, 4).astype(int)
kappa_preds = apply_thresholds(all_preds, kappa_optimized_threshold)
false_positive_preds = apply_thresholds(all_preds, false_positive_optimized_threshold)

cm_c = confusion_matrix(all_labels, classic_preds)
cm_k = confusion_matrix(all_labels, kappa_preds)
cm_fp = confusion_matrix(all_labels, false_positive_preds)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, cohen_kappa_score

labels = all_labels
preds_cont = all_preds


kappa_c = cohen_kappa_score(labels, classic_preds, weights='quadratic')
kappa_k = cohen_kappa_score(labels, kappa_preds, weights='quadratic')
kappa_m = cohen_kappa_score(labels, false_positive_preds, weights='quadratic')

# Configurazione Grafica
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 6))
class_names = [f'Grado {i}' for i in range(5)]

sns.heatmap(cm_c, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_title(f'Ottimizzazione Max Kappa\n(Quadratic Kappa: {kappa_c:.4f})')
ax1.set_xlabel('Predetto')
ax1.set_ylabel('Reale')


sns.heatmap(cm_k, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_title(f'Ottimizzazione Max Kappa\n(Quadratic Kappa: {kappa_k:.4f})')
ax2.set_xlabel('Predetto')
ax2.set_ylabel('Reale')


sns.heatmap(cm_fp, annot=True, fmt='d', cmap='Greens', 
            xticklabels=class_names, yticklabels=class_names, ax=ax3)
ax3.set_title(f'Ottimizzazione False Positive\n(Quadratic Kappa: {kappa_m:.4f})')
ax3.set_xlabel('Predetto')
ax3.set_ylabel('Reale')

plt.tight_layout()
plt.show()

# --- Report Testuale ---
print("\n" + "="*50)
print("ANALISI COMPARATIVA DELLE SOGLIE")
print("="*50)
